# Pengujian Prediction Request ke Model di Cloud

Notebook ini mengirim *prediction request* ke model TF Serving yang berjalan di Railway. Signature `serving_default` menerima `tf.Example` mentah (di-*encode* base64), sehingga preprocessing dijalankan di dalam model.

In [ ]:
import base64

import requests
import tensorflow as tf

BASE_URL = "https://obesity-model-serving-production.up.railway.app"
MODEL_NAME = "obesity-model"

# Urutan mengikuti serving_model/<versi>/assets/label_vocab
LABELS = [
    "Obesity_Type_I", "Obesity_Type_III", "Overweight_Level_I",
    "Obesity_Type_II", "Normal_Weight", "Overweight_Level_II",
    "Insufficient_Weight",
]

## 1. Cek status model

In [ ]:
response = requests.get(f"{BASE_URL}/v1/models/{MODEL_NAME}", timeout=30)
print(response.status_code)
print(response.json())

## 2. Membuat request

Semua fitur mentah harus dikirim (termasuk `Height` dan `Weight`; keduanya diabaikan model, tetapi ada di skema data).

In [ ]:
def build_example(row):
    """Mengubah dict fitur menjadi tf.Example ter-serialisasi."""
    feature = {}
    for key, value in row.items():
        if isinstance(value, str):
            feature[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[value.encode()]))
        else:
            feature[key] = tf.train.Feature(
                float_list=tf.train.FloatList(value=[float(value)]))
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()


sample = {
    "Gender": "Female", "Age": 21.0, "Height": 1.62, "Weight": 64.0,
    "family_history_with_overweight": "yes", "FAVC": "no", "FCVC": 2.0,
    "NCP": 3.0, "CAEC": "Sometimes", "SMOKE": "no", "CH2O": 2.0,
    "SCC": "no", "FAF": 0.0, "TUE": 1.0, "CALC": "no",
    "MTRANS": "Public_Transportation",
}

payload = {"instances": [{"examples": {"b64": base64.b64encode(
    build_example(sample)).decode()}}]}

## 3. Mengirim prediction request

In [ ]:
response = requests.post(
    f"{BASE_URL}/v1/models/{MODEL_NAME}:predict", json=payload, timeout=30)
print(response.status_code)
probabilities = response.json()["predictions"][0]
for label, prob in sorted(zip(LABELS, probabilities), key=lambda x: -x[1]):
    print(f"{label:22s} {prob:.4f}")

**Catatan:** urutan `LABELS` mengikuti berkas `serving_model/<versi>/assets/label_vocab` (baris ke-n = indeks ke-n pada keluaran model). Jika model dilatih ulang, urutan ini bisa berubah, jadi periksa ulang berkas tersebut.